# Creación de Word Embeddings desde Cero  

## Fundamentos Teóricos – Skip-Gram con Negative Sampling

---

### 1 Motivación

Las redes neuronales no pueden procesar texto directamente.  
Necesitan representaciones numéricas.

Una representación básica es el **one-hot encoding**:

$$
x \in \mathbb{R}^{V}
$$

donde:

- ($V$) = tamaño del vocabulario  
- Solo una posición es 1  
- El resto son 0  

Problema:

$$
\|x_i - x_j\| = \sqrt{2}
$$

para cualquier par de palabras distintas.

No existe información semántica.

---

### 2 Objetivo de los Embeddings

Queremos aprender una función:

$$
f: \text{palabra} \rightarrow \mathbb{R}^d
$$

tal que palabras con contextos similares tengan vectores cercanos en el espacio vectorial.

---

### 3 Modelo Skip-Gram

Dada una palabra central ($w_t$), queremos predecir sus palabras de contexto.

Función objetivo:

$$
\max \sum_{t=1}^{T} \sum_{k=-c}^{c} \log P(w_{t+k} \mid w_t)
$$

donde:

- ($c$) = tamaño de la ventana  
- ($T$) = número total de palabras en el corpus  

---

### 4 Representación Paramétrica

Sea:

- ($V$) = tamaño del vocabulario  
- ($d$) = dimensión del embedding  

Definimos dos matrices:

$$
W^{(in)} \in \mathbb{R}^{V \times d}
$$

$$
W^{(out)} \in \mathbb{R}^{V \times d}
$$

Cada palabra tiene:

- Un vector de entrada ($v_w$)
- Un vector de salida ($u_w$)

---

### 5 Softmax Completo

La probabilidad condicional es:

$$
P(w_o \mid w_i)
=
\frac{\exp(u_{w_o}^\top v_{w_i})}
{\sum_{w=1}^{V} \exp(u_w^\top v_{w_i})}
$$

Problema:  
El denominador requiere recorrer todo el vocabulario ($V$).

Costo computacional:

$$
O(V)
$$

---

### 6 Negative Sampling

Para evitar el costo del softmax completo, se usa una aproximación binaria.

Maximizamos:

$$
\log \sigma(u_o^\top v_i)
+
\sum_{k=1}^{K}
\log \sigma(-u_{n_k}^\top v_i)
$$

donde:

- ($n_k$) = palabras negativas muestreadas  
- ($K$) = número de negativos  
- ($\sigma(x) = \frac{1}{1+e^{-x}}$)

Interpretación:

- Palabras reales → producto escalar alto  
- Palabras aleatorias → producto escalar bajo  

---

### 7 Función de Pérdida

Para un par positivo ($w_i, w_o$):

$$
\lambda
=
- \log \sigma(u_o^\top v_i)
-
\sum_{k=1}^{K}
\log \sigma(-u_{n_k}^\top v_i)
$$

---

### 8 Gradientes

Sea:

$$
x = u_o^\top v_i
$$

Entonces:

$$
\frac{\partial \lambda}{\partial x}
=
\sigma(x) - 1
$$

Para negativos:

$$
\frac{\partial \lambda}{\partial x_{neg}}
=
\sigma(x_{neg})
$$

---

### 9 Actualización por Descenso de Gradiente

$$
\theta \leftarrow \theta - \eta \nabla_\theta \lambda
$$

donde:

- ($\eta$) = learning rate  
- ($\theta$) = parámetros del modelo  

---

### 10 Interpretación Geométrica

El producto escalar:

$$
u_w^\top v_c
$$

mide la alineación entre vectores.

Si es alto:

$$
\Rightarrow \text{Alta probabilidad de co-ocurrencia}
$$

Si es bajo:

$$
\Rightarrow \text{Baja probabilidad}
$$

---

### 11 ¿Por qué emerge semántica?

Si dos palabras tienen distribuciones de contexto similares:

$$
P(context \mid word_1)
\approx
P(context \mid word_2)
$$

entonces reciben gradientes similares  
y sus vectores convergen a posiciones cercanas.

---

### 12 Relación con PMI

Word2Vec aproxima implícitamente:

$$
PMI(w, c) = \log \frac{P(w,c)}{P(w)P(c)}
$$

Es equivalente a factorizar una matriz de co-ocurrencias.

---

### 13 Resumen del Algoritmo

1. Construir vocabulario  
2. Inicializar matrices aleatorias  
3. Generar pares centro-contexto  
4. Calcular producto escalar  
5. Aplicar sigmoid  
6. Calcular pérdida  
7. Actualizar parámetros  

---

### Conclusión

Los embeddings no almacenan definiciones.

Aprenden la **geometría estadística del lenguaje**.

Esa geometría emerge únicamente de patrones de co-ocurrencia en grandes corpus.

# Import

In [ ]:
import numpy as np
from collections import Counter

# Semilla para reproducibilidad
np.random.seed(7)

# Base de Datos Ejemplo

In [ ]:
# 1) BD de ejemplo
vocab_20 = [
    "perro", "gato", "animal", "casa", "calle",
    "come", "bebe", "carne", "leche", "agua",
    "mesa", "silla", "cocina", "jardin", "parque",
    "corre", "duerme", "juega", "feliz", "triste"
]

# BD (lista de "frases")
sentences = [
    "perro corre parque feliz",
    "gato duerme casa feliz",
    "perro come carne agua",
    "gato bebe leche agua",
    "animal corre jardin parque",
    "casa cocina mesa silla",
    "gato juega casa feliz",
    "perro juega calle parque",
    "animal duerme casa triste",
    "perro gato animal",
    "mesa silla cocina casa",
    "perro bebe agua feliz",
    "gato come carne triste",
    "jardin parque calle",
]

# Tokenizamos
corpus = [s.split() for s in sentences]

# Verificación: que todas las palabras estén en las 20 definidas
all_tokens = [w for sent in corpus for w in sent]
unknown = sorted(set(all_tokens) - set(vocab_20))
if unknown:
    raise ValueError(f"Hay palabras fuera del vocab_20: {unknown}")

# Mapeo de Palabras

In [ ]:
# 2) Mapeo palabra <-> índice (V = 20)
word2idx = {w: i for i, w in enumerate(vocab_20)}
idx2word = {i: w for w, i in word2idx.items()}
V = len(vocab_20)  # 20

# Convertimos corpus a índices
corpus_idx = [[word2idx[w] for w in sent] for sent in corpus]

In [ ]:
corpus_idx

[[0, 15, 14, 18],
 [1, 16, 3, 18],
 [0, 5, 7, 9],
 [1, 6, 8, 9],
 [2, 15, 13, 14],
 [3, 12, 10, 11],
 [1, 17, 3, 18],
 [0, 17, 4, 14],
 [2, 16, 3, 19],
 [0, 1, 2],
 [10, 11, 12, 3],
 [0, 6, 9, 18],
 [1, 5, 7, 19],
 [13, 14, 4]]

# One Hot (Como era antes)

In [ ]:
def one_hot(palabra, word2idx, V):
    """Crear vector one-hot para una palabra."""
    vector = np.zeros(V)
    vector[word2idx[palabra]] = 1
    return vector

# Mostrar one-hot para algunas palabras
palabras_ejemplo = ["gato", "perro", "casa", "come"]
print("One-Hot Encoding:")
print(f"{'Palabra':<10} Vector ({V} dimensiones)")
print("-" * 60)
for p in palabras_ejemplo:
    vec = one_hot(p, word2idx, V)
    print(f"{p:<10} {vec.astype(int)}")

# Calcular similitud coseno entre one-hot vectors
from numpy.linalg import norm

def similitud_coseno(v1, v2):
    """Calcular similitud coseno entre dos vectores."""
    denominador = norm(v1) * norm(v2)
    if denominador == 0:
        return 0.0
    return np.dot(v1, v2) / denominador

print("\nSimilitud coseno entre one-hot vectors:")
print(f"  gato vs perro: {similitud_coseno(one_hot('gato', word2idx, V), one_hot('perro', word2idx, V)):.2f}")
print(f"  gato vs casa:  {similitud_coseno(one_hot('gato', word2idx, V), one_hot('casa', word2idx, V)):.2f}")
print("\n-> Problema: TODAS las similitudes son 0! No captura relaciones semanticas.")

One-Hot Encoding:
Palabra    Vector (20 dimensiones)
------------------------------------------------------------
gato       [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
perro      [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
casa       [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
come       [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Similitud coseno entre one-hot vectors:
  gato vs perro: 0.00
  gato vs casa:  0.00

-> Problema: TODAS las similitudes son 0! No captura relaciones semanticas.


# Construcción pares

In [ ]:
# 3) Construir pares (center, context) con ventana de contexto
window = 2  # mira 2 palabras a la izquierda y derecha

pairs = []
for sent in corpus_idx:
    for i, center in enumerate(sent):
        left = max(0, i - window)
        right = min(len(sent), i + window + 1)
        for j in range(left, right):
            if j == i:
                continue
            context = sent[j]
            pairs.append((center, context))

pairs = np.array(pairs, dtype=np.int64)
print(f"Pares (center, context) creados: {len(pairs)}")

Pares (center, context) creados: 132


In [ ]:
pairs[:10]

array([[ 0, 15],
       [ 0, 14],
       [15,  0],
       [15, 14],
       [15, 18],
       [14,  0],
       [14, 15],
       [14, 18],
       [18, 15],
       [18, 14]])

# Distribución para Negative Sampling

In [ ]:
# 4) Distribución para Negative Sampling (unigram^0.75)
counts = Counter(all_tokens)
freq = np.array([counts[w] for w in vocab_20], dtype=np.float64)

# Distribución de muestreo negativa (Mikolov): p(w) ∝ f(w)^0.75
p_neg = freq ** 0.75
p_neg = p_neg / p_neg.sum()

def sample_negatives(K, forbidden_idx):
    """
    Muestra K negativos según p_neg evitando forbidden_idx (center o context).
    """
    negs = []
    while len(negs) < K:
        cand = np.random.choice(V, p=p_neg)
        if cand != forbidden_idx:
            negs.append(cand)
    return np.array(negs, dtype=np.int64)

# Modelo

In [ ]:
# 5) Modelo: dos matrices de embeddings (input y output)
#    - W_in:  V x d
#    - W_out: V x d

d = 10  # dimensión del embedding (puedes subirla a 50 si quieres)
W_in = 0.01 * np.random.randn(V, d) # Se inicia con valores random
W_out = 0.01 * np.random.randn(V, d) # Se inicia con valores random

In [ ]:
W_in[0]

array([ 1.69052570e-02, -4.65937371e-03,  3.28201637e-04,  4.07516283e-03,
       -7.88923029e-03,  2.06557291e-05, -8.90385858e-06, -1.75472431e-02,
        1.01765801e-02,  6.00498516e-03])

In [ ]:
W_out[0]

array([-0.01683397,  0.01015015, -0.01441544, -0.01343629, -0.00361668,
        0.00788505,  0.00300114,  0.00810768, -0.00443644, -0.00172183])

# Entrenamiento

In [ ]:
# Sigmoid y entrenamiento 1 paso
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def train_step(center, context, K=5, lr=0.05):
    """
    Un update de Skip-gram con Negative Sampling:
    - positivo: (center -> context) con label 1
    - negativos: (center -> neg_i) con label 0
    """
    v_c = W_in[center]          # (d,)
    u_o = W_out[context]        # (d,)

    # --------- POSITIVO ----------
    score_pos = np.dot(u_o, v_c)
    p_pos = sigmoid(score_pos)  # probabilidad del positivo
    # gradiente para maximizar log(sigmoid(score_pos)):
    # dL/dscore = (p_pos - 1)
    g_pos = (p_pos - 1.0)

    # Gradientes
    grad_vc = g_pos * u_o
    grad_uo = g_pos * v_c

    # --------- NEGATIVOS ----------
    negs = sample_negatives(K, forbidden_idx=context)
    u_negs = W_out[negs]  # (K, d)

    scores_neg = u_negs @ v_c              # (K,)
    p_neg_scores = sigmoid(scores_neg)     # (K,)
    # Para negativos maximizamos log(sigmoid(-score)):
    # equivalente a minimizar log(sigmoid(score)) con label 0
    # dL/dscore = p_neg_scores
    g_negs = p_neg_scores                  # (K,)

    # acumular gradientes
    grad_vc += (g_negs[:, None] * u_negs).sum(axis=0)  # (d,)
    grad_unegs = g_negs[:, None] * v_c                 # (K, d)

    # --------- UPDATE ----------
    W_in[center]   -= lr * grad_vc
    W_out[context] -= lr * grad_uo
    W_out[negs]    -= lr * grad_unegs

    # pérdida (solo para monitoreo)
    # loss = -log(sigmoid(pos)) - sum(log(sigmoid(-neg)))
    loss_pos = -np.log(p_pos + 1e-12)
    loss_neg = -np.log(1.0 - p_neg_scores + 1e-12).sum()
    return loss_pos + loss_neg

In [ ]:
# 7) Entrenamiento (epochs sobre los pares)
epochs = 200
K = 5
lr = 0.05

for ep in range(1, epochs + 1):
    # barajar pares
    idx_perm = np.random.permutation(len(pairs))
    total_loss = 0.0
    for idx in idx_perm:
        c, o = pairs[idx]
        total_loss += train_step(c, o, K=K, lr=lr)

    if ep % 20 == 0:
        print(f"Epoch {ep:3d} | loss promedio = {total_loss / len(pairs):.4f}")

Epoch  20 | loss promedio = 2.6877
Epoch  40 | loss promedio = 2.1289
Epoch  60 | loss promedio = 2.0096
Epoch  80 | loss promedio = 1.9809
Epoch 100 | loss promedio = 1.8217
Epoch 120 | loss promedio = 1.8589
Epoch 140 | loss promedio = 1.8901
Epoch 160 | loss promedio = 2.0423
Epoch 180 | loss promedio = 1.8182
Epoch 200 | loss promedio = 1.9522


# Embedding

In [ ]:
# 8) Embeddings finales
#    Usaremos W_in como embedding (también se puede promediar W_in y W_out)
E = W_in  # (20, d)

def cosine_sim(a, b):
    na = np.linalg.norm(a) + 1e-12
    nb = np.linalg.norm(b) + 1e-12
    return float(np.dot(a, b) / (na * nb))

def most_similar(word, topk=5):
    w_idx = word2idx[word]
    v = E[w_idx]
    sims = []
    for i in range(V):
        if i == w_idx:
            continue
        sims.append((idx2word[i], cosine_sim(v, E[i])))
    sims.sort(key=lambda x: x[1], reverse=True)
    return sims[:topk]

In [ ]:
E.shape

(20, 10)

In [ ]:
E[0]

array([ 0.80196146,  0.34613254,  1.11510914,  1.76811817, -0.57863765,
       -1.0274686 , -0.45399116,  0.54391811,  1.71730574,  0.31301542])

In [ ]:
tests = ["perro", "gato", "agua", "mesa", "parque", "feliz", "triste"]
for w in tests:
    print(f"\nSimilares a '{w}':")
    for ww, s in most_similar(w, topk=5):
        print(f"  {ww:>8s}  sim={s:.3f}")


Similares a 'perro':
    jardin  sim=0.909
     leche  sim=0.870
     carne  sim=0.823
    triste  sim=0.813
      agua  sim=0.810

Similares a 'gato':
      agua  sim=0.897
    triste  sim=0.885
     feliz  sim=0.776
     silla  sim=0.770
     corre  sim=0.768

Similares a 'agua':
      gato  sim=0.897
      bebe  sim=0.853
    triste  sim=0.852
     leche  sim=0.838
    parque  sim=0.826

Similares a 'mesa':
    cocina  sim=0.882
     silla  sim=0.871
    jardin  sim=0.854
      casa  sim=0.846
     feliz  sim=0.835

Similares a 'parque':
    jardin  sim=0.853
      agua  sim=0.826
      mesa  sim=0.823
     calle  sim=0.819
    animal  sim=0.796

Similares a 'feliz':
      mesa  sim=0.835
     leche  sim=0.821
     silla  sim=0.815
    triste  sim=0.813
     calle  sim=0.804

Similares a 'triste':
      gato  sim=0.885
      agua  sim=0.852
      come  sim=0.842
     perro  sim=0.813
     feliz  sim=0.813
